# Linear Regression

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import joblib

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA n'est pas disponible. Verifiez les drivers NVIDIA, la presence du GPU et l'environnement du notebook."
    )

device = torch.device("cuda")
print(f"Device utilise: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Ouvre le CSV ../data/reddit_depression_dataset_cleaned_v2.csv
df = pd.read_csv("../data/reddit_depression_dataset_cleaned_v2.csv")

In [ ]:
# 1. Split stratifié (même répartition train/test)
X = df["clean_text"]
y = df["label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# 2. Vectorisation TF-IDF (5000 features, bi-grammes)
X_train_clean = X_train.fillna("").astype(str)
X_test_clean = X_test.fillna("").astype(str)

vectorizer = TfidfVectorizer(
    max_features=5000, stop_words="english", ngram_range=(1, 2), min_df=5
)
X_train_tfidf = vectorizer.fit_transform(X_train_clean)
X_test_tfidf = vectorizer.transform(X_test_clean)

In [ ]:
# 3. Modèle avec gestion déséquilibre (class_weight)
model = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight="balanced",  # ⚠️ CLÉ pour le déséquilibre
)
model.fit(X_train_tfidf, y_train)

In [ ]:
# 4. Évaluation
y_pred = model.predict(X_test_tfidf)
print("Classification Report :\n", classification_report(y_test, y_pred))
print("Confusion Matrix :\n", confusion_matrix(y_test, y_pred))

In [ ]:
# 5. Sauvegarde (pour RNCP BC02)
joblib.dump(model, "../models/depression_classifier.pkl")
joblib.dump(vectorizer, "../models/tfidf_vectorizer.pkl")
print("✅ Modèle baseline sauvegardé !")

In [ ]:
# Charger et tester
loaded_model = joblib.load("../models/depression_classifier.pkl")
loaded_vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")

In [ ]:
test_text = (
    "I feel very unhappy and hopeless about my life"  # Exemple de texte à tester
)
test_vec = loaded_vectorizer.transform([test_text])
pred = loaded_model.predict(test_vec)[0]
proba = loaded_model.predict_proba(test_vec)[0]

print(f"Texte : {test_text}")
print(f"Prédiction : {'Dépressif' if pred == 1 else 'Non dépressif'}")
print(f"Probabilité dépressif : {proba[1]:.2%}")

| Modèle   | F1 Dépressifs | Recall Dépressifs | Precision | Accuracy |
| -------- | ------------- | ----------------- | --------- | -------- |
| Logistic | 83%           | 91%               | 76%       | 93%      |
| XGBoost  | 80%           | 87%               | 73%       | 91%      |

Points forts :

Recall 91% sur dépressifs : détecte 9/10 posts dépressifs (crucial en santé mentale)

Precision 76% : quand il dit "dépressif", c’est juste 3 fois sur 4

F1 83% : excellent compromis